In [1]:
# 先导入keras库
from keras.datasets import cifar10
# 调用加载数据的函数加载数据
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

170498071/170498071 [==============================] - 143s 1us/step
(50000, 32, 32, 3)
(10000, 32, 32, 3)
(50000, 1)
(10000, 1)


In [2]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255
print(X_train[:,5,:])

[[[0.2784314  0.1882353  0.11372549]
  [0.32941177 0.20784314 0.09411765]
  [0.43137255 0.28627452 0.14509805]
  ...
  [0.2627451  0.15294118 0.05882353]
  [0.13725491 0.03921569 0.        ]
  [0.1254902  0.05098039 0.01568628]]

 [[0.53333336 0.5647059  0.58431375]
  [0.53333336 0.58431375 0.64705884]
  [0.49803922 0.5803922  0.6431373 ]
  ...
  [0.2627451  0.27058825 0.22352941]
  [0.31764707 0.3254902  0.2784314 ]
  [0.35686275 0.3647059  0.31764707]]

 [[1.         1.         1.        ]
  [1.         1.         1.        ]
  [1.         1.         1.        ]
  ...
  [1.         1.         1.        ]
  [1.         1.         1.        ]
  [1.         1.         1.        ]]

 ...

 [[0.7647059  0.9019608  0.972549  ]
  [0.6313726  0.8509804  0.9764706 ]
  [0.57254905 0.85490197 0.9843137 ]
  ...
  [0.24313726 0.27058825 0.33333334]
  [0.04313726 0.07450981 0.07450981]
  [0.02352941 0.07058824 0.03921569]]

 [[0.8627451  0.9254902  0.9882353 ]
  [0.85882354 0.91764706 0.9764706 ]


In [3]:
from keras.utils import np_utils  
Y_train = np_utils.to_categorical(y_train, 10)
Y_test = np_utils.to_categorical(y_test, 10)
print(Y_train)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]]


In [34]:
datagen = ImageDataGenerator(
    rotation_range=30,           # 增大旋转范围
    width_shift_range=0.2,       # 增大平移范围
    height_shift_range=0.2,      # 增大平移范围
    horizontal_flip=True,
    vertical_flip=True,          # 新增垂直翻转
    zoom_range=0.2,              # 增大缩放范围
    shear_range=0.2,             # 新增剪切变换
    fill_mode='nearest',
    brightness_range=[0.7, 1.3]  # 新增亮度变化
)
datagen.fit(X_train)

In [35]:
from keras.models import Sequential  
from keras.layers import Dense, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.regularizers import l2  # 新增 L2 正则化导入
from keras.layers import BatchNormalization, Dropout  # 新增批量归一化和 Dropout 导入
model = Sequential()

In [36]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, BatchNormalization, Dropout
from keras.regularizers import l2

model = Sequential()

# 第一层卷积块（滤波器数量增加）
model.add(Conv2D(64, (3,3), padding='same', input_shape=X_train.shape[1:], kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(64, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# 第二层卷积块（滤波器数量增加）
model.add(Conv2D(128, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(128, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# 第三层卷积块（滤波器数量增加）
model.add(Conv2D(256, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(256, (3,3), padding='same', kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# 全连接层
model.add(Flatten())
model.add(Dense(512, kernel_regularizer=l2(0.0005)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

In [37]:
from keras.optimizers import Adam  # 新增导入

model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.0005),  # 显式设置学习率并降低
    metrics=['accuracy']
)

In [38]:
# 第26个单元格
from keras.optimizers import Adam

model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),  # 提高初始学习率
    metrics=['accuracy']
)
# 在这里添加模型结构打印
model.summary()

from keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    datagen.flow(X_train, Y_train, batch_size=128),
    steps_per_epoch=len(X_train) // 128,
    epochs=50,
    validation_data=(X_test, Y_test),
    callbacks=[early_stopping, reduce_lr]
)

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_14 (Conv2D)          (None, 32, 32, 64)        1792      
                                                                 
 batch_normalization_17 (Bat  (None, 32, 32, 64)       256       
 chNormalization)                                                
                                                                 
 activation_18 (Activation)  (None, 32, 32, 64)        0         
                                                                 
 conv2d_15 (Conv2D)          (None, 32, 32, 64)        36928     
                                                                 
 batch_normalization_18 (Bat  (None, 32, 32, 64)       256       
 chNormalization)                                                
                                                                 
 activation_19 (Activation)  (None, 32, 32, 64)       

ImportError: Using brightness shifts requires PIL. Install PIL or Pillow.

In [39]:
score = model.evaluate(X_test, Y_test)
print('Test loss: ', score[0])
print('Test accuracy: ', score[1])

313/313 [==============================] - 18s 55ms/step - loss: 3.1124 - accuracy: 0.0995
Test loss:  3.112396001815796
Test accuracy:  0.09950000047683716
